## VUILD110 data generation for benchmarking all the tools

In [ ]:
import pickle
import sys
from datasets import load_from_disk, concatenate_datasets
sys.path.append("/scratch/project_465001027/Spatialformer/utils")
from utils import *
import scanpy as sc

sample_name = "VUILD110"

#loading the 
combined_dataset = load_from_disk("/scratch/project_465001027/Spatialformer/cache/xenium_pandavid_dataset4")  
index_path = "/scratch/project_465001027/Spatialformer/data/sample_cell_index.pkl"
sample_cell_index = get_index(combined_dataset, save_file = index_path)
combined_dataset_all = concatenate_datasets([combined_dataset["train"], combined_dataset["test"], combined_dataset["validation"]])
sample_dataset = combined_dataset_all.select(list(sample_cell_index[sample_name].values()))
sparse_adj, cell_ids = get_adj(sample_dataset, radius = 30, plot = False, sym = False)

anns = sample_dataset["Annotations"]
niches = sample_dataset["Niche_Annotations"]
rows, cols = sparse_adj.nonzero()
cts = ["Macrophages", "T-cells"]
# Iterate over these indices
adj_cell_ids = [] 
for i, j in tqdm(zip(rows, cols)):
#     print(f"Non-zero entry at row {i}, column {j}")
    # Perform any operations you need here
    if ((anns[i] == "Macrophages") & (anns[j] == "T-cells"))|((anns[j] == "Macrophages") & (anns[i] == "T-cells")):
        adj_cell_ids.append(cell_ids[i])
        adj_cell_ids.append(cell_ids[j])
        
unique_paired_mac_t = np.unique(adj_cell_ids)
ct_dict = dict(zip(cell_ids, anns))
niche_dict = dict(zip(cell_ids, niches))
pickle.dump(ct_dict ,open(f"/scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/data/{sample_name}_ct_dict.pkl", "wb"))
pickle.dump(niche_dict ,open(f"/scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/data/{sample_name}_niche_dict.pkl", "wb"))
pickle.dump(unique_paired_mac_t, open(f"/scratch/project_465001027/Spatialformer/downstream/cell_cell_communication/data/{sample_name}_unique_paired_mac_t.pkl", "wb"))

In [ ]:
sample_dataset

Building the X for single sample

In [ ]:
def getexpcount(dataset, label_type):
    # Assume 'train_dataset' is a pandas DataFrame or a similar object like a list of dictionaries
    # First, extract the relevant columns
    expression_data = [i[0] for i in dataset['Expression']]  # This should be a list of expression values per cell
    gene_names = dataset[0]['Gene']  # This should be a list of corresponding gene names
    cell_ids = dataset["Cell_id"]
    df = pd.DataFrame(expression_data, index=cell_ids, columns=gene_names)
    if label_type == "cell_types":
        anns = dataset["Annotations"]
    elif label_type == "cell_niches":
        anns = dataset["Niche_Annotations"]
    return df, np.array(anns)

In [ ]:
VUILD110_X, VUILD110_ann = getexpcount(sample_dataset, "cell_types")
VUILD110_X, VUILD110_niche = getexpcount(sample_dataset, "cell_niches")

# Split the data into train test val for benchmarking

In [ ]:
VUILD110_X["Annotations"] = VUILD110_ann
VUILD110_X["Niche_Annotations"] = VUILD110_niche
#group by cell type to subset the dataset
VUILD110_X = VUILD110_X[VUILD110_X["Annotations"] != "not available"]
VUILD110_X = VUILD110_X[VUILD110_X["Niche_Annotations"] != "not available"]


In [ ]:
#cell type split
def split_dataset(df, ann):
    np.random.seed(42) #each time should get the same proportion
    df_train_X = df.groupby(ann).apply(lambda x: x.sample(frac=0.8)).reset_index(level=0, drop=True)
    df_temp_X = df.drop(df_train_X.index)
    df_test_X = df_temp_X.groupby(ann).apply(lambda x: x.sample(frac=0.5)).reset_index(level=0, drop=True)
    df_val_X = df_temp_X.drop(df_test_X.index)
    train_ann = np.array(df_train_X[ann])
    df_train_X.drop(columns=["Annotations","Niche_Annotations"], inplace=True)
    
    test_ann = np.array(df_test_X[ann])
    df_test_X.drop(columns=["Annotations","Niche_Annotations"], inplace=True)
    
    val_ann = np.array(df_val_X[ann])
    df_val_X.drop(columns=["Annotations","Niche_Annotations"], inplace=True)
    
    return df_train_X, df_test_X, df_val_X, train_ann, test_ann, val_ann
    
ct_train_X, ct_test_X, ct_val_X, train_ct, test_ct, val_ct = split_dataset(VUILD110_X, "Annotations")
nc_train_X, nc_test_X, nc_val_X, train_nc, test_nc, val_nc = split_dataset(VUILD110_X, "Niche_Annotations")

#saving X
ct_train_X.to_csv("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_train_X.csv")
ct_test_X.to_csv("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_test_X.csv")
ct_val_X.to_csv("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_val_X.csv")
 
    
nc_train_X.to_csv("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_train_X.csv")
nc_test_X.to_csv("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_test_X.csv")
nc_val_X.to_csv("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_val_X.csv")
 
    
#saving label
np.save("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_train.npy", train_ct)
np.save("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_test.npy", test_ct)
np.save("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_ct_val.npy", val_ct)

np.save("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_train.npy", train_nc)
np.save("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_test.npy", test_nc)
np.save("/scratch/project_465001027/Spatialformer/downstream/cell_types_nich_annotation/data/VUILD110_nc_val.npy", val_nc)
